# AI数据集清洗与排序工具实现

本实验实现AI数据集的生成、清洗、排序和统计功能。


## 一、生成模拟AI训练数据

构造150条模拟传感器数据，包含数值型数据（含少量缺失值、异常值）。


In [ ]:
import csv
import random
import math

random.seed(42)

num_records = 150
data = []

for i in range(num_records):
    sensor_id = f"S{i+1:03d}"
    timestamp = f"2024-01-01 12:{i//60:02d}:{i%60:02d}"
    
    temperature = round(random.gauss(25, 3), 2)
    if random.random() < 0.05:
        temperature = None
    elif random.random() < 0.03:
        temperature = round(random.gauss(25, 15), 2)
    
    humidity = round(random.gauss(60, 8), 2)
    if random.random() < 0.05:
        humidity = None
    elif random.random() < 0.03:
        humidity = round(random.gauss(60, 30), 2)
    
    pressure = round(random.gauss(1013, 10), 2)
    if random.random() < 0.05:
        pressure = None
    elif random.random() < 0.03:
        pressure = round(random.gauss(1013, 50), 2)
    
    vibration = round(random.gauss(0.5, 0.1), 3)
    if random.random() < 0.05:
        vibration = None
    elif random.random() < 0.03:
        vibration = round(random.gauss(0.5, 0.5), 3)
    
    data.append([sensor_id, timestamp, temperature, humidity, pressure, vibration])

with open('ai_sensor_data.csv', 'w', newline='', encoding='utf-8') as f:
    writer = csv.writer(f)
    writer.writerow(['sensor_id', 'timestamp', 'temperature', 'humidity', 'pressure', 'vibration'])
    for row in data:
        writer.writerow(row)

print(f"生成了 {num_records} 条模拟数据，已保存到 ai_sensor_data.csv")
print("前10条数据预览：")
for row in data[:10]:
    print(row)


## 二、数据集清洗

1. 直接删除缺失值的记录
2. 修正异常值（超出该列数据均值±2倍标准差的值，用均值替换）


In [ ]:
def read_csv_file(filename):
    data = []
    with open(filename, 'r', encoding='utf-8') as f:
        reader = csv.reader(f)
        header = next(reader)
        for row in reader:
            data.append(row)
    return header, data

def delete_missing_values(data):
    cleaned = []
    for row in data:
        has_missing = False
        for val in row[2:]:
            if val == '' or val is None:
                has_missing = True
                break
        if not has_missing:
            cleaned.append(row)
    return cleaned

def fix_outliers(data, numeric_cols):
    numeric_data = []
    for row in data:
        nums = []
        for col in numeric_cols:
            nums.append(float(row[col]))
        numeric_data.append(nums)

    means = []
    stds = []
    for col_idx in range(len(numeric_cols)):
        values = [row[col_idx] for row in numeric_data]
        mean = sum(values) / len(values)
        variance = sum((x - mean) ** 2 for x in values) / len(values)
        std = math.sqrt(variance)
        means.append(mean)
        stds.append(std)

    fixed_data = []
    for row in data:
        new_row = row.copy()
        for i, col in enumerate(numeric_cols):
            val = float(new_row[col])
            if val < means[i] - 2 * stds[i] or val > means[i] + 2 * stds[i]:
                new_row[col] = str(round(means[i], 2))
        fixed_data.append(new_row)

    return fixed_data

header, raw_data = read_csv_file('ai_sensor_data.csv')
print(f"原始数据行数: {len(raw_data)}")

cleaned_no_missing = delete_missing_values(raw_data)
print(f"删除缺失值后行数: {len(cleaned_no_missing)}")

numeric_cols = [2, 3, 4, 5]
cleaned_data = fix_outliers(cleaned_no_missing, numeric_cols)
print(f"修正异常值后行数: {len(cleaned_data)}")

print("\n清洗后前10条数据：")
for row in cleaned_data[:10]:
    print(row)


## 三、记录排序

对清洗后的数据集记录按照数据值从小到大的顺序进行排序，并将排序结果写入CSV文件。


In [ ]:
def sort_data(data, sort_col=2):
    return sorted(data, key=lambda x: float(x[sort_col]))

sorted_data = sort_data(cleaned_data, sort_col=2)

with open('sorted_ai_data.csv', 'w', newline='', encoding='utf-8') as f:
    writer = csv.writer(f)
    writer.writerow(header)
    for row in sorted_data:
        writer.writerow(row)

print("排序后数据已保存到 sorted_ai_data.csv")
print("\n排序后前10条数据（按温度升序）：")
for row in sorted_data[:10]:
    print(row)

print("\n排序后最后10条数据：")
for row in sorted_data[-10:]:
    print(row)


## 四、数据统计

计算清洗后数据的均值、中位数、标准差、最大值、最小值。


In [ ]:
def calculate_statistics(data, numeric_cols, col_names):
    stats = {}
    for i, col in enumerate(numeric_cols):
        values = [float(row[col]) for row in data]
        values.sort()
        
        mean = sum(values) / len(values)
        
        n = len(values)
        if n % 2 == 0:
            median = (values[n//2 - 1] + values[n//2]) / 2
        else:
            median = values[n//2]
        
        variance = sum((x - mean) ** 2 for x in values) / len(values)
        std = math.sqrt(variance)
        
        min_val = min(values)
        max_val = max(values)
        
        stats[col_names[i]] = {
            '均值': round(mean, 2),
            '中位数': round(median, 2),
            '标准差': round(std, 2),
            '最小值': round(min_val, 2),
            '最大值': round(max_val, 2)
        }
    return stats

col_names = ['temperature', 'humidity', 'pressure', 'vibration']
stats = calculate_statistics(cleaned_data, numeric_cols, col_names)

print("=" * 60)
print("清洗后数据统计结果")
print("=" * 60)
for col_name, stat in stats.items():
    print(f"\n{col_name}:")
    print(f"  均值: {stat['均值']}")
    print(f"  中位数: {stat['中位数']}")
    print(f"  标准差: {stat['标准差']}")
    print(f"  最小值: {stat['最小值']}")
    print(f"  最大值: {stat['最大值']}")
print("=" * 60)
